In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)

# ============================================================
# SKILLBRIDGE DATASET
# 50,000 realistic student profiles
# Based ONLY on the information our SkillBridge profile collects
# ============================================================

N = 50000

data = pd.DataFrame({
    "CGPA": np.round(np.random.normal(7.5, 1.0, N).clip(5.0, 10.0), 2),

    "Semester": np.random.randint(1, 9, N),

    "Skills_Count": np.random.randint(1, 11, N),

    "Projects_Count": np.random.choice(
        [0, 1, 2, 3, 4, 5],
        N,
        p=[0.05, 0.15, 0.25, 0.25, 0.20, 0.10]
    ),

    "Internship": np.random.choice(
        [0, 1],
        N,
        p=[0.65, 0.35]
    ),

    "Coding_Profile": np.random.choice(
        [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        N,
        p=[0.05, 0.05, 0.07, 0.08, 0.10,
           0.12, 0.13, 0.13, 0.11, 0.09, 0.07]
    ),

    "Aptitude_Score": np.round(
        np.random.normal(65, 18, N).clip(0, 100), 1
    ),

    "Communication_Score": np.round(
        np.random.normal(65, 17, N).clip(0, 100), 1
    ),

    "Resume_Score": np.round(
        np.random.normal(65, 18, N).clip(0, 100), 1
    )
})


# ============================================================
# FEATURE SCORES
# ============================================================

data["Academic_Score"] = (data["CGPA"] / 10) * 100

data["Skill_Score"] = data["Skills_Count"] * 10

data["Project_Score"] = data["Projects_Count"] * 20

data["Internship_Score"] = data["Internship"] * 100

data["Coding_Score"] = data["Coding_Profile"] * 10


# ============================================================
# REALISTIC PLACEMENT READINESS
# ============================================================

base_score = (
    data["Academic_Score"] * 0.20 +
    data["Skill_Score"] * 0.15 +
    data["Project_Score"] * 0.15 +
    data["Internship_Score"] * 0.10 +
    data["Coding_Score"] * 0.10 +
    data["Aptitude_Score"] * 0.10 +
    data["Communication_Score"] * 0.10 +
    data["Resume_Score"] * 0.10
)

# Add realistic variation so the target is not a perfect
# mathematical copy of the input features.
noise = np.random.normal(0, 6, N)

data["Placement_Readiness"] = (
    base_score + noise
).clip(0, 100).round(2)


# ============================================================
# FINAL DATASET
# ============================================================

features = [
    "CGPA",
    "Semester",
    "Skills_Count",
    "Projects_Count",
    "Internship",
    "Coding_Profile",
    "Aptitude_Score",
    "Communication_Score",
    "Resume_Score"
]

X = data[features]
y = data["Placement_Readiness"]

print("Dataset shape:", data.shape)
print()
print("Number of students:", len(data))
print()
print("Model features:")
print(features)
print()
print("Target: Placement_Readiness")
print()
print("Placement Readiness statistics:")
print(data["Placement_Readiness"].describe())
print()
print("First 5 students:")
display(data[features + ["Placement_Readiness"]].head())

Dataset shape: (50000, 15)

Number of students: 50000

Model features:
['CGPA', 'Semester', 'Skills_Count', 'Projects_Count', 'Internship', 'Coding_Profile', 'Aptitude_Score', 'Communication_Score', 'Resume_Score']

Target: Placement_Readiness

Placement Readiness statistics:
count    50000.000000
mean        59.843116
std         10.701767
min         16.050000
25%         52.460000
50%         59.780000
75%         67.090000
max        100.000000
Name: Placement_Readiness, dtype: float64

First 5 students:


,CGPA,Semester,Skills_Count,Projects_Count,Internship,Coding_Profile,Aptitude_Score,Communication_Score,Resume_Score,Placement_Readiness
0,8.00,8,1,5,0,1,78.6,75.9,42.0,66.83
1,7.36,3,3,3,0,4,65.8,84.0,50.7,44.98
2,8.15,4,6,3,1,8,59.5,88.7,80.2,71.24
3,9.02,5,7,5,0,5,52.8,84.9,86.6,68.22
4,7.27,2,10,3,1,8,49.3,52.4,82.4,75.87


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# Split the SkillBridge dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

# Create the model
placement_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

# Train
placement_model.fit(X_train, y_train)

# Predict
y_pred = placement_model.predict(X_test)

# Evaluate
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\nModel trained successfully!")
print("Mean Absolute Error:", round(mae, 2))
print("R² Score:", round(r2, 4))

Training samples: 40000
Testing samples: 10000

Model trained successfully!
Mean Absolute Error: 5.02
R² Score: 0.6628


In [4]:
# Feature importance
importance = pd.Series(
    placement_model.feature_importances_,
    index=features
).sort_values(ascending=False)

print("SkillBridge Feature Importance:")
display(importance.to_frame("Importance"))

SkillBridge Feature Importance:


,Importance
Internship,0.246935
Skills_Count,0.203206
Projects_Count,0.172346
Coding_Profile,0.088261
CGPA,0.076469
Resume_Score,0.069774
Aptitude_Score,0.065716
Communication_Score,0.064441
Semester,0.012852


In [5]:
import os
import joblib

os.makedirs("../models", exist_ok=True)

model_path = "../models/placement_readiness_model.pkl"

joblib.dump(
    {
        "model": placement_model,
        "features": features
    },
    model_path
)

print("Model saved successfully!")
print("Location:", model_path)

Model saved successfully!
Location: ../models/placement_readiness_model.pkl


In [6]:
# Test SkillBridge with one sample student

sample_student = pd.DataFrame([{
    "CGPA": 8.5,
    "Semester": 6,
    "Skills_Count": 7,
    "Projects_Count": 3,
    "Internship": 1,
    "Coding_Profile": 8,
    "Aptitude_Score": 75,
    "Communication_Score": 80,
    "Resume_Score": 78
}])

# Predict Placement Readiness
readiness_prediction = placement_model.predict(sample_student)[0]

# Keep between 0 and 100
readiness_prediction = np.clip(readiness_prediction, 0, 100)

print("Placement Readiness:", round(readiness_prediction, 2), "%")

Placement Readiness: 73.71 %


In [7]:
# Save our final SkillBridge ML dataset

dataset_path = "../data/skillbridge_placement_readiness_dataset.csv"

data.to_csv(dataset_path, index=False)

print("SkillBridge dataset saved successfully!")
print("Location:", dataset_path)
print("Dataset shape:", data.shape)

SkillBridge dataset saved successfully!
Location: ../data/skillbridge_placement_readiness_dataset.csv
Dataset shape: (50000, 15)


In [8]:
import joblib

model_path = "../models/placement_readiness_model.pkl"

saved_model = joblib.load(model_path)

print("Saved model loaded successfully!")
print("Available features:")
print(saved_model["features"])

Saved model loaded successfully!
Available features:
['CGPA', 'Semester', 'Skills_Count', 'Projects_Count', 'Internship', 'Coding_Profile', 'Aptitude_Score', 'Communication_Score', 'Resume_Score']
